# OpenArm — MeanFlow v2 (2 camera depth) trên Colab

1. Runtime → Change runtime type → **GPU**.
2. Dataset nằm ở **Được chia sẻ với tôi › vrbt › dataset › 2026-10-13**. Thư mục "được chia sẻ" không tự hiện trong `MyDrive`:
   vào Drive → chuột phải thư mục `vrbt` → *Sắp xếp / Thêm lối tắt vào Drive* → chọn *Drive của tôi*. Hoặc sửa `RAW` bên dưới.
3. Chỉnh tham số ở ô **Tham số**, rồi chạy lần lượt các ô.

In [ ]:
#@title Tham số
#@markdown ### Nguồn
REPO_URL = 'https://github.com/vanphong864/openarm_act.git' #@param {type:"string"}
REPO_REF = 'main' #@param {type:"string"}
RAW_DIR = '/content/drive/MyDrive/vrbt/dataset/2026-10-13' #@param {type:"string"}
OUT_ROOT = '/content/drive/MyDrive/vrbt/meanflow_v2' #@param {type:"string"}
RUN_NAME = 'run1' #@param {type:"string"}
#@markdown ### Convert (đổi giá trị -> tự convert lại vào zarr mới)
HEIGHT = 120 #@param {type:"integer"}
WIDTH = 160 #@param {type:"integer"}
ACTION_SOURCE = 'commands' #@param ["commands", "next_qpos", "qpos"]
VAL_FRACTION = 0.15 #@param {type:"number"}
MAX_EPISODES = 0 #@param {type:"integer"}
#@markdown ### Model
DOWN_DIMS = '64,128,256' #@param {type:"string"}
HORIZON = 16 #@param {type:"integer"}
N_OBS_STEPS = 4 #@param {type:"integer"}
N_ACTION_STEPS = 4 #@param {type:"integer"}
FEATURE_DIM = 96 #@param {type:"integer"}
#@markdown ### Train
STEPS = 20000 #@param {type:"integer"}
BATCH_SIZE = 16 #@param {type:"integer"}
LR = 0.0001 #@param {type:"number"}
WEIGHT_DECAY = 0.0001 #@param {type:"number"}
WARMUP_STEPS = 500 #@param {type:"integer"}
SAVE_EVERY = 500 #@param {type:"integer"}
NUM_WORKERS = 2 #@param {type:"integer"}
SEED = 42 #@param {type:"integer"}
RESUME = True #@param {type:"boolean"}
RUN_SMOKE_FIRST = True #@param {type:"boolean"}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, json, yaml, hashlib
from pathlib import Path

RAW = Path(RAW_DIR); RUN = Path(OUT_ROOT) / RUN_NAME; WORK = Path('/content/openarm_act')
assert RAW.is_dir(), f'Không thấy {RAW}. Thêm lối tắt thư mục vrbt vào My Drive hoặc sửa RAW_DIR.'
n_h5 = sum(1 for _ in RAW.rglob('*.hdf5')); assert n_h5, f'Không có file .hdf5 trong {RAW}'
print('HDF5 files:', n_h5)

def sh(cmd, **kw): subprocess.run(cmd, check=True, **kw)
def run_live(cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in p.stdout: print(line, end='', flush=True)
        rc = p.wait()
    except BaseException:
        p.terminate(); p.wait(); raise
    if rc: raise subprocess.CalledProcessError(rc, cmd)

if not WORK.exists(): sh(['git', 'clone', REPO_URL, str(WORK)])
sh(['git', '-C', str(WORK), 'fetch', 'origin', REPO_REF])
sh(['git', '-C', str(WORK), 'checkout', '-q', '--detach', 'FETCH_HEAD'])
print('Commit:', subprocess.check_output(['git', '-C', str(WORK), 'rev-parse', '--short', 'HEAD'], text=True).strip())
os.chdir(WORK); sys.path.insert(0, str(WORK / 'src')); os.environ['PYTHONPATH'] = str(WORK / 'src')

In [ ]:
%pip install -q -r requirements-colab.txt
import torch
assert torch.cuda.is_available(), 'Chưa bật GPU'
print('Torch', torch.__version__, '|', torch.cuda.get_device_name(0))

## Convert HDF5 → zarr
Zarr ghi ở `/content` (nhanh), cache tar lưu lên Drive; tên cache phụ thuộc tham số convert.

In [ ]:
import zarr
tag = hashlib.sha1(json.dumps([HEIGHT, WIDTH, ACTION_SOURCE, VAL_FRACTION, SEED, MAX_EPISODES, n_h5]).encode()).hexdigest()[:8]
DATASET = Path(f'/content/openarm_{tag}.zarr'); ARCHIVE = Path(OUT_ROOT) / 'cache' / f'openarm_{tag}.zarr.tar'
def is_complete(p):
    try: return bool(zarr.open_group(str(p), mode='r').attrs.get('complete'))
    except Exception: return False
if DATASET.exists() and not is_complete(DATASET): sh(['rm', '-rf', str(DATASET)])
if not DATASET.exists():
    if ARCHIVE.exists():
        print('Khôi phục cache', ARCHIVE); sh(['tar', '-xf', str(ARCHIVE), '-C', '/content'])
    else:
        cmd = [sys.executable, 'scripts/convert_meanflow_dataset.py', '--raw', str(RAW), '--output', str(DATASET),
               '--height', str(HEIGHT), '--width', str(WIDTH), '--action-source', ACTION_SOURCE,
               '--val-fraction', str(VAL_FRACTION), '--seed', str(SEED), '--skip-invalid']
        if MAX_EPISODES: cmd += ['--max-episodes', str(MAX_EPISODES)]
        run_live(cmd)
        ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
        sh(['tar', '-cf', str(ARCHIVE), '-C', '/content', DATASET.name])
meta = dict(zarr.open_group(str(DATASET), mode='r').attrs)
assert meta.get('complete')
print('Episodes:', len(meta['episodes']), '| skipped:', len(meta['skipped']),
      '| train/val:', sum(e['split']=='train' for e in meta['episodes']), '/', sum(e['split']=='val' for e in meta['episodes']))

## Cấu hình & train
Khi resume, tham số model/train đã lưu được giữ nguyên (trừ `STEPS`). Muốn đổi tham số → đổi `RUN_NAME`.

In [ ]:
CONFIG = WORK / 'conf/colab_run.yaml'
if RESUME and (RUN / 'last.pt').exists():
    cfg = yaml.safe_load((RUN / 'train_config.yaml').read_text()); print('Resume: dùng cấu hình đã lưu')
else:
    cfg = yaml.safe_load((WORK / 'conf/meanflow_v2.yaml').read_text())
    cfg['seed'] = SEED
    cfg['model'].update(down_dims=[int(x) for x in DOWN_DIMS.split(',')], horizon=HORIZON, n_obs_steps=N_OBS_STEPS,
                        n_action_steps=N_ACTION_STEPS, feature_dim=FEATURE_DIM)
    cfg['train'].update(steps=STEPS, batch_size=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY,
                        warmup_steps=WARMUP_STEPS, save_every=SAVE_EVERY, num_workers=NUM_WORKERS)
CONFIG.write_text(yaml.safe_dump(cfg)); print(CONFIG.read_text())
from openarm_flow.meanflow import MeanFlowPolicy
m = MeanFlowPolicy(**cfg['model']); print('Parameters:', f'{sum(p.numel() for p in m.parameters()):,}'); del m

In [ ]:
if RUN_SMOKE_FIRST and not (RUN / 'last.pt').exists():
    import copy, tempfile
    s = copy.deepcopy(cfg); s['train'].update(steps=2, batch_size=2, save_every=1, val_samples_per_episode=2, num_workers=0)
    sc = WORK / 'conf/colab_smoke.yaml'; sc.write_text(yaml.safe_dump(s))
    out = Path(tempfile.mkdtemp(dir='/content')) / 'run'
    run_live([sys.executable, '-u', 'scripts/train_meanflow.py', '--dataset', str(DATASET), '--output', str(out),
              '--config', str(sc), '--device', 'cuda'])
    print('Smoke OK')

In [ ]:
cmd = [sys.executable, '-u', 'scripts/train_meanflow.py', '--dataset', str(DATASET), '--output', str(RUN),
       '--config', str(CONFIG), '--device', 'cuda', '--steps', str(cfg['train']['steps'])]
if RESUME and (RUN / 'last.pt').exists(): cmd.append('--resume')
elif RUN.exists() and any(RUN.iterdir()): raise SystemExit(f'{RUN} đã có dữ liệu nhưng không có last.pt/RESUME=False: đổi RUN_NAME.')
RUN.mkdir(parents=True, exist_ok=True)
run_live(cmd)

## Đánh giá & tải model

In [ ]:
run_live([sys.executable, 'scripts/eval_meanflow.py', '--dataset', str(DATASET), '--checkpoint', str(RUN / 'best_release'),
          '--device', 'cuda', '--samples-per-episode', '128', '--batch-size', str(cfg['train']['batch_size']),
          '--output', str(RUN / 'evaluation.json')])
from IPython.display import Image, display
for f in ('loss_curve.png', 'action_metrics.png'):
    if (RUN / f).exists(): display(Image(filename=str(RUN / f)))

In [ ]:
import shutil
from google.colab import files
files.download(shutil.make_archive('/content/meanflow_best_release', 'zip', root_dir=str(RUN), base_dir='best_release'))